# Paligemma 3b Nocaps Baseline

This notebook was reorganized for the GitHub reproducibility package.
Original file: `NoCaps-Baseline/PaliGemma3.ipynb`.

**Security note:** hard-coded Hugging Face tokens were removed. Use interactive login or environment variables instead.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
!pip -q uninstall -y datasets
!pip -q install -q "datasets<4.0.0"


In [ ]:
from datasets import load_dataset

ds_val = load_dataset("HuggingFaceM4/NoCaps", split="validation")
print("len:", len(ds_val))
print("cols:", ds_val.column_names)

r0 = ds_val[0]
print("file:", r0["image_file_name"])
display(r0["image"])


In [ ]:
import json

GT_PATH = "/content/drive/MyDrive/datasets/nocaps/nocaps_val_4500_captions_domain_norm.json"
gt = json.load(open(GT_PATH, "r", encoding="utf-8"))

fname_to_domain = {im["file_name"]: im["domain_norm"] for im in gt["images"]}
fname_to_gtid   = {im["file_name"]: int(im["id"]) for im in gt["images"]}

in_idx, near_idx, out_idx = [], [], []
miss = 0

for i, r in enumerate(ds_val):
    fn = r["image_file_name"]
    d = fname_to_domain.get(fn)
    if d is None:
        miss += 1
        continue
    if d == "in":
        in_idx.append(i)
    elif d == "near":
        near_idx.append(i)
    elif d == "out":
        out_idx.append(i)

ds_in   = ds_val.select(in_idx)
ds_near = ds_val.select(near_idx)
ds_out  = ds_val.select(out_idx)

print("hf split lens:", len(ds_in), len(ds_near), len(ds_out), "miss:", miss)


In [ ]:
!pip -q install -U pycocotools
!pip -q install -U git+https://github.com/salaniz/pycocoevalcap


In [ ]:
i = 4499
print(ds_val[i]["image_file_name"])
display(ds_val[i]["image"])


In [ ]:
import os
from tqdm import tqdm

OUT_DIR = "/content/drive/MyDrive/datasets/nocaps/images_val_hf"
os.makedirs(OUT_DIR, exist_ok=True)

for r in tqdm(ds_val, total=len(ds_val)):
    fn = r["image_file_name"]
    path = os.path.join(OUT_DIR, fn)
    if not os.path.exists(path):
        r["image"].save(path, format="JPEG")

print("saved to:", OUT_DIR)


In [ ]:
import os, glob

def folder_report(path):
    exts = ["*.jpg","*.jpeg","*.png","*.webp","*.bmp","*.gif"]
    all_files = [f for f in glob.glob(os.path.join(path, "**", "*"), recursive=True) if os.path.isfile(f)]
    img_files = []
    for ext in exts:
        img_files += glob.glob(os.path.join(path, "**", ext), recursive=True)

    total_size = sum(os.path.getsize(f) for f in all_files)
    img_size = sum(os.path.getsize(f) for f in img_files)

    print("\n📁", path)
    print("  total files:", len(all_files))
    print("  image files:", len(img_files))
    print("  total size (MB):", round(total_size/1024/1024, 2))
    print("  image size (MB):", round(img_size/1024/1024, 2))
    print("  sample:", [os.path.basename(f) for f in sorted(img_files)[:5]])

base = "/content/drive/MyDrive/datasets/nocaps"
folder_report(os.path.join(base, "images_val_hf"))
folder_report(os.path.join(base, "val"))


In [ ]:
import os, json

GT_PATH = "/content/drive/MyDrive/datasets/nocaps/nocaps_val_4500_captions_domain_norm.json"
IMG_DIR = "/content/drive/MyDrive/datasets/nocaps/images_val_hf"

gt = json.load(open(GT_PATH, "r", encoding="utf-8"))
gt_files = [im["file_name"] for im in gt["images"]]

exists = sum(1 for fn in gt_files if os.path.exists(os.path.join(IMG_DIR, fn)))
print("GT file_name match in images_val_hf:", exists, "/", len(gt_files))


In [ ]:
import json
from collections import Counter

GT_PATH = "/content/drive/MyDrive/datasets/nocaps/nocaps_val_4500_captions_domain_norm.json"
gt = json.load(open(GT_PATH, "r", encoding="utf-8"))

c = Counter(im["domain_norm"] for im in gt["images"])
total = len(gt["images"])

print("Toplam:", total)
for k in ["in", "near", "out", "unknown"]:
    n = c.get(k, 0)
    print(f"{k:7s}: {n:4d}  ({n/total*100:.2f}%)")


# **PALIGEMMA 448 Denemesi**

In [ ]:
# 1. Pillow sürümü uyumluluğu (COCO dosyasındaki gibi)
!pip uninstall -y pillow
!pip install "pillow<10.0.0"

# 2. Transformers'ı GitHub'dan çekiyoruz (PaliGemma için kritik)
!pip install -q git+https://github.com/huggingface/transformers.git
!pip install -q accelerate bitsandbytes peft
!pip install -q pycocotools
!pip install -q git+https://github.com/salaniz/pycocoevalcap

print("✅ Kurulumlar tamamlandı. Lütfen yukarıdan 'Runtime -> Restart Session' yapın.")

In [ ]:
import torch
from transformers import AutoProcessor, PaliGemmaForConditionalGeneration
from huggingface_hub import login
from PIL import Image
import os
import json
from tqdm import tqdm

# Buraya kendi HF tokenını gir (COCO notebook'unda girdiğin gibi)
# login(token="HF_TOKEN_BURAYA")

MODEL_ID = "google/paligemma-3b-ft-cococap-448"
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"⏳ {MODEL_ID} yükleniyor...")

# AutoProcessor kullanımı (COCO dosyasındaki gibi)
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = PaliGemmaForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
).eval()

print("✅ Model ve Processor Hazır!")

In [ ]:
# Dosya yolları
GT_PATH = "/content/drive/MyDrive/datasets/nocaps/nocaps_val_4500_captions_domain_norm.json"
IMG_DIR = "/content/drive/MyDrive/datasets/nocaps/images_val_hf"

# Ground Truth dosyasını yükle
with open(GT_PATH, "r", encoding="utf-8") as f:
    nocaps_gt = json.load(f)

# İşlenecek görsellerin listesi
images_to_process = nocaps_gt['images']

print(f"📂 Görsel Klasörü: {IMG_DIR}")
print(f"🖼️ Toplam Görsel Sayısı: {len(images_to_process)}")

In [ ]:
# Dosya yolları
GT_PATH = "/content/drive/MyDrive/datasets/nocaps/nocaps_val_4500_captions_domain_norm.json"
IMG_DIR = "/content/drive/MyDrive/datasets/nocaps/images_val_hf"

with open(GT_PATH, "r", encoding="utf-8") as f:
    nocaps_gt = json.load(f)

results = []
PROMPT = "caption en"

print(f"🚀 {len(nocaps_gt['images'])} görsel için tahmin başlıyor...")

for img_info in tqdm(nocaps_gt['images']):
    image_path = os.path.join(IMG_DIR, img_info['file_name'])

    if not os.path.exists(image_path):
        continue

    try:
        image = Image.open(image_path).convert("RGB")

        # TEZ İÇİN KRİTİK NOKTA: Input formatı ve Beam Search
        # COCO dosyasında "<image>" tokenını elle eklemişsin, burada da ekliyoruz.
        inputs = processor(text="<image>" + PROMPT, images=image, return_tensors="pt").to(device)

        with torch.no_grad():
            output = model.generate(
                **inputs,
                max_new_tokens=100,
                do_sample=False,
                num_beams=5   # COCO dosyasındaki parametre (Standart: 1)
            )

        decoded = processor.batch_decode(output, skip_special_tokens=True)[0]
        # Prompt kısmını temizleme
        caption = decoded.replace(PROMPT, "").strip().replace("\n", "")

        results.append({
            "image_id": img_info['id'],
            "caption": caption
        })

    except Exception as e:
        print(f"Hata ({img_info['file_name']}): {e}")

# Sonuçları kaydet
with open("paligemma_nocaps_results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print("✅ İşlem tamamlandı ve kaydedildi.")

In [ ]:
from transformers import InstructBlipProcessor, InstructBlipForConditionalGeneration
import torch
from PIL import Image
import requests

model = InstructBlipForConditionalGeneration.from_pretrained("Salesforce/instructblip-flan-t5-xl")
processor = InstructBlipProcessor.from_pretrained("Salesforce/instructblip-flan-t5-xl")

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

url = "https://raw.githubusercontent.com/salesforce/LAVIS/main/docs/_static/Confusing-Pictures.jpg"
image = Image.open(requests.get(url, stream=True).raw).convert("RGB")
prompt = "What is unusual about this image?"
inputs = processor(images=image, text=prompt, return_tensors="pt").to(device)

outputs = model.generate(
        **inputs,
        do_sample=False,
        num_beams=5,
        max_length=256,
        min_length=1,
        top_p=0.9,
        repetition_penalty=1.5,
        length_penalty=1.0,
        temperature=1,
)
generated_text = processor.batch_decode(outputs, skip_special_tokens=True)[0].strip()
print(generated_text)


In [ ]:
# 1. Yarım kalan indirmeleri ve cache'i temizle
!rm -rf pycocoevalcap
!rm -rf /root/.cache/huggingface
!rm -rf /content/pycocoevalcap

# 2. Java (Hafif sürüm) ve Kütüphaneyi sıfırdan kur
!apt-get install -y openjdk-8-jdk-headless -qq > /dev/null
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"

!pip uninstall -y pycocoevalcap
!pip install -q git+https://github.com/salaniz/pycocoevalcap

print("✅ Temizlik yapıldı. SPICE iptal edildi. Skorlamaya geçebilirsin.")

In [ ]:
import os
import sys
import json
import urllib.request

# ==============================================================================
# 1. TEMİZLİK VE MANUEL KURULUM (Garantili Yöntem)
# ==============================================================================
print("🛠️ Ortam temizleniyor ve 'Manuel Kurulum' yapılıyor...")

# Eski kurulumları uçur
os.system("pip uninstall -y pycocoevalcap")
os.system("rm -rf pycocoevalcap")

# Kütüphaneyi GitHub'dan yerel klasöre çek
if not os.path.exists("pycocoevalcap"):
    os.system("git clone https://github.com/salaniz/pycocoevalcap.git")

# Java Kurulumu (Emin olmak için)
os.system("apt-get install -y openjdk-8-jdk-headless -qq > /dev/null")
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"

# ==============================================================================
# 2. JAVA DOSYASINI ZORLA İNDİRME (ContentTooShort Hatasının Çözümü)
# ==============================================================================
print("⬇️ Tokenizer dosyası manuel indiriliyor (Python downloader bypass edildi)...")

jar_url = "http://www.stanford.edu/npm/stanford-corenlp-3.4.1.jar"
# Alternatif güvenilir mirror (Stanford sunucusu yavaşsa)
backup_url = "https://repo1.maven.org/maven2/edu/stanford/nlp/stanford-corenlp/3.4.1/stanford-corenlp-3.4.1.jar"

target_dir = "pycocoevalcap/tokenizer"
target_file = os.path.join(target_dir, "stanford-corenlp-3.4.1.jar")

# Klasör yoksa oluştur
os.makedirs(target_dir, exist_ok=True)

# WGET ile indir (Python'dan çok daha sağlamdır)
# Eğer dosya zaten varsa silip tekrar indiriyoruz ki bozuk kalmasın
if os.path.exists(target_file):
    os.remove(target_file)

ret = os.system(f"wget {backup_url} -O {target_file}")

if ret != 0:
    print("⚠️ Maven mirror başarısız, Stanford sunucusu deneniyor...")
    os.system(f"wget {jar_url} -O {target_file}")

if os.path.exists(target_file):
    print(f"✅ JAR dosyası başarıyla indirildi: {os.path.getsize(target_file) / 1024 / 1024:.2f} MB")
else:
    raise Exception("❌ JAR dosyası indirilemedi! İnternet bağlantını kontrol et.")

# Yolu sisteme ekle ki Python bu klasörü görsün
sys.path.append(os.path.abspath("pycocoevalcap"))

# ==============================================================================
# 3. DEĞERLENDİRME (Local Kütüphane ile)
# ==============================================================================
from tokenizer.ptbtokenizer import PTBTokenizer
from bleu.bleu import Bleu
from rouge.rouge import Rouge
from cider.cider import Cider

# Dosya Yolları
GT_PATH = "/content/drive/MyDrive/datasets/nocaps/nocaps_val_4500_captions_domain_norm.json"
RES_PATH = "paligemma_nocaps_results.json"

print("\n📊 NoCaps Değerlendirmesi Başlıyor...")

# Verileri Yükle
coco = json.load(open(GT_PATH))
cocoRes = json.load(open(RES_PATH))

# Format Düzenleme (Manuel çağırdığımız için ID'leri eşleştirmeliyiz)
# GT formatı: {img_id: [{'caption': ...}]}
gts = {}
for img in coco['images']:
    gts[img['id']] = []
for ann in coco['annotations']:
    gts[ann['image_id']].append(ann)

# Res formatı: {img_id: [{'caption': ...}]}
res = {}
for ann in cocoRes:
    res[ann['image_id']] = [ann]

# Domain Ayrımı
ids_in   = [img['id'] for img in coco['images'] if img['domain_norm'] == 'in']
ids_near = [img['id'] for img in coco['images'] if img['domain_norm'] == 'near']
ids_out  = [img['id'] for img in coco['images'] if img['domain_norm'] == 'out']
ids_all  = [img['id'] for img in coco['images']]

def evaluate_manual(img_ids, title):
    if not img_ids: return
    print(f"\n{'='*10} {title} ({len(img_ids)}) {'='*10}")

    # Sadece ilgili ID'leri filtrele
    gts_curr = {i: gts[i] for i in img_ids if i in gts}
    res_curr = {i: res[i] for i in img_ids if i in res}

    # 1. TOKENIZATION (Artık manuel indirdiğimiz JAR ile yapılacak)
    print("   Tokenization yapılıyor...")
    tokenizer = PTBTokenizer()
    gts_tok = tokenizer.tokenize(gts_curr)
    res_tok = tokenizer.tokenize(res_curr)

    # 2. SKORLAMA
    scorers = [
        (Bleu(4), ["Bleu_1", "Bleu_2", "Bleu_3", "Bleu_4"]),
        (Rouge(), "ROUGE_L"),
        (Cider(), "CIDEr")
    ]

    for scorer, method in scorers:
        score, scores = scorer.compute_score(gts_tok, res_tok)
        if isinstance(method, list):
            for m, s in zip(method, score):
                print(f"{m:10s}: {s:.3f}")
        else:
            print(f"{method:10s}: {score:.3f}")

# Çalıştır
evaluate_manual(ids_in, "IN-DOMAIN")
evaluate_manual(ids_near, "NEAR-DOMAIN")
evaluate_manual(ids_out, "OUT-OF-DOMAIN")
evaluate_manual(ids_all, "OVERALL")